In [ ]:
!pip install datasets huggingface_hub pandas

import pandas as pd
import os
import re
from datasets import Dataset, Audio
from huggingface_hub import login
from typing import Callable, Tuple

# Login to Hugging Face (Will prompt for token)
print("Logging into Hugging Face...")
login()

# ----------------------------
# Core lexicon (Egyptian-ish)
# ----------------------------

_AR_DIGITS = {
    0: "صفر", 1: "واحد", 2: "اتنين", 3: "تلاتة", 4: "أربعة", 5: "خمسة",
    6: "ستة", 7: "سبعة", 8: "تمانية", 9: "تسعة", 10: "عشرة",
    11: "حداشر", 12: "اتناشر", 13: "تلتاشر", 14: "أربعتاشر", 15: "خمستاشر",
    16: "ستاشر", 17: "سبعتاشر", 18: "تمانتاشر", 19: "تسعتاشر"
}

_AR_TENS = {
    20: "عشرين", 30: "تلاتين", 40: "أربعين", 50: "خمسين",
    60: "ستين", 70: "سبعين", 80: "تمانين", 90: "تسعين"
}

_AR_HUNDREDS = {
    100: "مية", 200: "ميتين", 300: "تلتمية", 400: "أربعمية",
    500: "خمسمية", 600: "ستمية", 700: "سبعمية",
    800: "تمنمية", 900: "تسعمية"
}

_SCALES = [
    (1_000_000_000, "مليار", "مليارين", "ملايير"),
    (1_000_000, "مليون", "مليونين", "ملايين"),
    (1_000, "ألف", "ألفين", "آلاف"),
]

_ARABIC_TO_WESTERN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

DECIMAL_WORD = "نقطة"
PERCENT_WORD = "في المية"

CURRENCY = {
    "EGP": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "ج": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "جنيه": ("جنيه", "جنيجنيههين", "جنيه", "قرش", "قرشين", "قروش"),
    "LE": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "£": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "$": ("دولار", "دولارين", "دولار", "سنت", "سنتين", "سنتات"),
    "USD": ("دولار", "دولارين", "دولار", "سنت", "سنتين", "سنتات"),
    "€": ("يورو", "يوروهين", "يوروهات", "سنت", "سنتين", "سنتات"),
    "EUR": ("يورو", "يوروهين", "يوروهات", "سنت", "سنتين", "سنتات"),
}

_AR_ORDINALS = {
    1: "الأول", 2: "التاني", 3: "التالت", 4: "الرابع", 5: "الخامس",
    6: "السادس", 7: "السابع", 8: "التامن", 9: "التاسع", 10: "العاشر",
    11: "الحداشر", 12: "الاتناشر"
}

_AR_MONTHS = {
    1: "يناير", 2: "فبراير", 3: "مارس", 4: "أبريل", 5: "مايو", 6: "يونيو",
    7: "يوليو", 8: "أغسطس", 9: "سبتمبر", 10: "أكتوبر", 11: "نوفمبر", 12: "ديسمبر"
}

# ----------------------------
# Helper functions
# ----------------------------

def normalize_arabic_digits(text: str) -> str:
    return text.translate(_ARABIC_TO_WESTERN)

def get_plural_form(n: int, singular: str, dual: str, plural: str) -> str:
    if n == 1:
        return singular
    elif n == 2:
        return dual
    else:
        return plural

# ----------------------------
# Number to Egyptian Arabic-ish
# ----------------------------

def int_to_egyptian_words(n: int) -> str:
    if n < 0:
        return "سالب " + int_to_egyptian_words(-n)
    if n < 20:
        return _AR_DIGITS[n]
    if n < 100:
        tens = (n // 10) * 10
        ones = n % 10
        if ones == 0:
            return _AR_TENS[tens]
        return f"{_AR_DIGITS[ones]} و{_AR_TENS[tens]}"
    if n < 1000:
        hundreds = (n // 100) * 100
        rest = n % 100
        if rest == 0:
            return _AR_HUNDREDS[hundreds]
        return f"{_AR_HUNDREDS[hundreds]} و{int_to_egyptian_words(rest)}"

    for scale_value, scale_singular, scale_dual, scale_plural in _SCALES:
        if n >= scale_value:
            major = n // scale_value
            rest = n % scale_value

            if major == 1:
                major_words = scale_singular
            elif major == 2:
                major_words = scale_dual
            elif major <= 10:
                major_words = f"{int_to_egyptian_words(major)} {scale_plural}"
            else:
                major_words = f"{int_to_egyptian_words(major)} {scale_singular}"

            if rest:
                major_words += f" و{int_to_egyptian_words(rest)}"
            return major_words

    return str(n)

def num_token_to_words(token: str) -> str:
    t = normalize_arabic_digits(token)
    t = t.replace("٬", "").replace(",", ".")

    if not re.fullmatch(r"-?\d+(\.\d+)?", t):
        return token

    neg = t.startswith("-")
    if neg:
        t = t[1:]

    if "." in t:
        a, b = t.split(".", 1)
        a_words = int_to_egyptian_words(int(a)) if a else "صفر"
        b_words = " ".join(_AR_DIGITS[int(ch)] for ch in b if ch.isdigit())
        out = f"{a_words} {DECIMAL_WORD} {b_words}".strip()
    else:
        out = int_to_egyptian_words(int(t))

    return ("سالب " + out) if neg else out

# ----------------------------
# Specific pattern normalizers
# ----------------------------

def normalize_percent(text: str) -> str:
    def repl(m):
        num = m.group("num")
        return f"{num_token_to_words(num)} {PERCENT_WORD}"
    return re.sub(r"(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*[%٪]+", repl, text)

def normalize_currency(text: str) -> str:
    def sym_first(m):
        sym = m.group("sym")
        num = m.group("num")

        if sym not in CURRENCY:
            return m.group(0)

        major_sg, major_du, major_pl, minor_sg, minor_du, minor_pl = CURRENCY[sym]
        t = normalize_arabic_digits(num).replace(",", ".")

        if "." in t:
            a, b = t.split(".", 1)
            a_i = int(a) if a else 0
            b2 = (b + "00")[:2]
            b_i = int(b2)

            major_word = get_plural_form(a_i, major_sg, major_du, major_pl)

            if b_i == 0:
                return f"{int_to_egyptian_words(a_i)} {major_word}"

            minor_word = get_plural_form(b_i, minor_sg, minor_du, minor_pl)
            return f"{int_to_egyptian_words(a_i)} {major_word} و{int_to_egyptian_words(b_i)} {minor_word}"

        a_i = int(t)
        major_word = get_plural_form(a_i, major_sg, major_du, major_pl)
        return f"{num_token_to_words(num)} {major_word}"

    text = re.sub(r"(?P<sym>[$€£])\s*(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)", sym_first, text)

    def num_first(m):
        num = m.group("num")
        cur = m.group("cur").strip()

        if cur not in CURRENCY:
            return m.group(0)

        major_sg, major_du, major_pl = CURRENCY[cur][:3]
        num_normalized = normalize_arabic_digits(num).replace(",", ".")

        if "." not in num_normalized:
            n = int(num_normalized)
            major_word = get_plural_form(n, major_sg, major_du, major_pl)
        else:
            major_word = major_sg

        return f"{num_token_to_words(num)} {major_word}"

    return re.sub(r"(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*(?P<cur>EGP|USD|EUR|LE|جنيه|ج)", num_first, text)

def normalize_dates(text: str) -> str:
    def repl_slash(m):
        day = int(normalize_arabic_digits(m.group("day")))
        month = int(normalize_arabic_digits(m.group("month")))
        year = int(normalize_arabic_digits(m.group("year"))) if m.group("year") else None

        day_word = _AR_ORDINALS.get(day, int_to_egyptian_words(day))
        month_word = _AR_MONTHS.get(month, int_to_egyptian_words(month))

        if year:
            year_word = int_to_egyptian_words(year)
            return f"{day_word} {month_word} {year_word}"
        return f"{day_word} {month_word}"

    text = re.sub(
        r"\b(?P<day>[\d٠-٩]{1,2})[/\-](?P<month>[\d٠-٩]{1,2})(?:[/\-](?P<year>[\d٠-٩]{2,4}))?\b",
        repl_slash,
        text
    )
    return text

def normalize_time(text: str) -> str:
    def repl(m):
        hh = int(normalize_arabic_digits(m.group("hh")))
        mm = int(normalize_arabic_digits(m.group("mm")))

        h_words = int_to_egyptian_words(hh)

        if mm == 0:
            return f"{h_words} بالظبط"
        if mm == 15:
            return f"{h_words} وربع"
        if mm == 30:
            return f"{h_words} ونص"
        if mm == 45:
            next_h = (hh + 1) % 24
            return f"{int_to_egyptian_words(next_h)} إلا ربع"

        m_words = int_to_egyptian_words(mm)
        if mm == 1:
            return f"{h_words} ودقيقة"
        elif mm == 2:
            return f"{h_words} ودقيقتين"
        else:
            return f"{h_words} و{m_words} دقيقة"

    return re.sub(r"\b(?P<hh>[0-2]?[\d٠-٩]):(?P<mm>[0-5][\d٠-٩])\b", repl, text)

def normalize_ranges(text: str) -> str:
    def repl(m):
        a, b = m.group("a"), m.group("b")
        return f"من {num_token_to_words(a)} لحد {num_token_to_words(b)}"

    return re.sub(
        r"\b(?P<a>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*[-–—]\s*(?P<b>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\b",
        repl,
        text
    )

def normalize_phone_like(text: str) -> str:
    def repl(m):
        s = normalize_arabic_digits(m.group(0))
        s = re.sub(r"\D", "", s)

        if len(s) < 7:
            return m.group(0)

        return " ".join(_AR_DIGITS[int(ch)] for ch in s)

    return re.sub(r"(\+?[\d٠-٩][\d٠-٩\s().\-]{6,}[\d٠-٩])", repl, text)

def normalize_plain_numbers(text: str) -> str:
    def repl(m):
        return num_token_to_words(m.group(0))

    return re.sub(
        r"(?<![A-Za-zا-ي])(-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)(?![A-Za-zا-ي])",
        repl,
        text
    )

def normalize_abbreviations(text: str) -> str:
    abbrev_map = {
        r"\bم\.": "متر",
        r"\bكم\.": "كيلومتر",
        r"\bكجم\.": "كيلوجرام",
        r"\bد\.": "دكتور",
        r"\bأ\.د\.": "أستاذ دكتور",
        r"\bش\.": "شارع",
        r"\bص\.ب\.": "صندوق بريد",
    }

    for pattern, replacement in abbrev_map.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text

def normalize_text_for_tts_egyptian(text: str) -> str:
    if not text or not text.strip():
        return ""

    text = text.strip()
    text = normalize_arabic_digits(text)

    text = normalize_dates(text)
    text = normalize_time(text)
    text = normalize_percent(text)
    text = normalize_currency(text)
    text = normalize_ranges(text)
    text = normalize_phone_like(text)
    text = normalize_abbreviations(text)
    text = normalize_plain_numbers(text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# ----------------------------
# Dataset Processing & Upload
# ----------------------------

base_dir =''                   # YOUR DATA
metadata_path = f'{base_dir}/metadata.csv'

print("Reading metadata from drive...")
df = pd.read_csv(metadata_path, sep='|', header=None, names=['audio_file', 'transcript'])
print(f"Total samples before cleaning: {len(df)}")

print("Applying text normalization...")
df['clean_transcript'] = df['transcript'].apply(normalize_text_for_tts_egyptian)

# Remove empty transcripts
df = df[df['clean_transcript'] != '']
df = df.dropna(subset=['clean_transcript'])

print("Validating audio files existence...")
valid_data = []
for index, row in df.iterrows():
    audio_path = os.path.join(base_dir, row['audio_file'])
    if os.path.exists(audio_path) and os.path.getsize(audio_path) > 0:
        valid_data.append({
            "audio": audio_path,
            "text": row['clean_transcript']
        })

valid_df = pd.DataFrame(valid_data)
print(f"Valid samples ready for upload: {len(valid_df)}")

print("Converting to Hugging Face Dataset format...")
hf_dataset = Dataset.from_pandas(valid_df)
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=24000))

# Hugging Face Repo details
hf_username = "eslamusamaaa"
dataset_name = "Elara-TTS-Data"
repo_id = f"{hf_username}/{dataset_name}"

print(f"Pushing dataset to https://huggingface.co/datasets/{repo_id} ...")
hf_dataset.push_to_hub(repo_id, private=True)

print("Dataset successfully cleaned and uploaded!")

Logging into Hugging Face...
Reading metadata from drive...
Total samples before cleaning: 18855
Applying text normalization...
Validating audio files existence...
Valid samples ready for upload: 18854
Converting to Hugging Face Dataset format...
Pushing dataset to https://huggingface.co/datasets/eslamusamaaa/Elara-TTS-Data ...


Uploading the dataset shards:   0%|          | 0/4 [00:00<?, ? shards/s]

Map:   0%|          | 0/4714 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/48 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/48 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|1         | 3.92MB /  387MB            

Dataset successfully cleaned and uploaded!


In [ ]:
from datasets import load_dataset
import IPython.display as ipd
import random

dataset = load_dataset("eslamusamaaa/Elara-TTS-Data", split="train", streaming=True)

sample = next(iter(dataset))

print(sample['text'])

ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate'])

حوالي نص المصريين الباليغين مصابين بالسمنة


In [ ]:
from datasets import load_dataset
from tqdm import tqdm


dataset = load_dataset("eslamusamaaa/Elara-TTS-Data", split="train", streaming=True)

total_seconds = 0
count = 0

for sample in tqdm(dataset, desc="Loading"):
    audio = sample["audio"]

    duration = len(audio["array"]) / audio["sampling_rate"]
    total_seconds += duration
    count += 1

total_hours = total_seconds / 3600
total_minutes = (total_seconds % 3600) / 60

print(f"\nإجمالي المقاطع اللي اتحسبت: {count} مقطع")
print(f"إجمالي المدة الزمنية الصافية: {int(total_hours)} ساعة و {int(total_minutes)} دقيقة")

Loading: 18854it [03:41, 85.19it/s] 


إجمالي المقاطع اللي اتحسبت: 18854 مقطع
إجمالي المدة الزمنية الصافية: 14 ساعة و 24 دقيقة
